# AML Rules and ML Triage MVP: Exploratory Data Analysis (EDA)

This notebook performs exploratory data analysis (EDA) on the IBM AML transaction dataset used for this project. It investigates amount distributions, categorical frequencies, temporal distributions, target class imbalance, and banking network properties.

## 1. Setup Environment and Configurations

We adjust paths to import the local packages and load the project YAML configurations.

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import json

# Adjust path to find the aml_mvp modules and config files
if Path.cwd().name == "notebooks":
    os.chdir("..")
if "src" not in sys.path:
    sys.path.append("src")

from aml_mvp.config import load_config
data_config = load_config("config/data_config.yaml")
print(f"Data Config Loaded for Dataset: {data_config['data']['dataset_name']}")

## 2. Global Dataset Statistics

We load and print the exact statistics of the full transaction dataset using the precomputed profile and quality reports.

In [ ]:
with open("outputs/metrics/data_profile.json", "r") as f:
    profile = json.load(f)
with open("outputs/metrics/data_quality_report.json", "r") as f:
    dq_report = json.load(f)

print("=== DATASET PROFILE ===")
print(f"Total Row Count:         {profile['row_count']:,}")
print(f"Laundering Alert Rate:   {profile['label_rate'] * 100:.5f}%")
print(f"Timeline Start:          {profile['date_min']}")
print(f"Timeline End:            {profile['date_max']}")
print(f"Average Trans. Amount:   ${profile['amount_summary']['mean']:.2f}")
print(f"Median Trans. Amount:    ${profile['amount_summary']['median']:.2f}")

print("\n=== DATA QUALITY METRICS ===")
print(f"Total Duplicates:        {dq_report.get('duplicate_count', 0):,}")
missing = dq_report.get('null_counts', {})
print("Missing Value Count per Column:")
for col, count in missing.items():
    print(f"  {col}: {count}")

## 3. Data Ingestion & Sampling

Since the dataset contains 6.9M rows, we load a representative sample of 200,000 transactions from the processed parquet files to perform high-speed visualizations.

In [ ]:
processed_path = "data/processed/base_transactions_with_splits.parquet"
print(f"Loading sample transactions from {processed_path}...")
sample_df = pd.read_parquet(processed_path)

# Stratified/Random sampling to limit memory footprint while plotting
sample_df = sample_df.sample(n=min(200000, len(sample_df)), random_state=42)
print(f"Loaded Sample Shape: {sample_df.shape}")

## 4. Class Imbalance Analysis

We inspect the laundering label distribution. Money laundering is a classic 'needle in a haystack' problem with extremely rare positives.

In [ ]:
import matplotlib.pyplot as plt

laundering_counts = sample_df["is_laundering"].value_counts()
print("=== TARGET DISTRIBUTION (SAMPLE) ===")
print(laundering_counts)
print(f"Laundering Rate: {laundering_counts.get(1, 0)/len(sample_df)*100:.5f}%")

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(["Non-Laundering", "Laundering"], laundering_counts.values, color=["#5D9CEC", "#FC6E51"], width=0.6)
ax.set_title("Laundering vs Non-Laundering Transactions", fontsize=12, fontweight='bold')
ax.set_ylabel("Record Count")
for i, val in enumerate(laundering_counts.values):
    ax.text(i, val + 500, f"{val:,} ({val/len(sample_df)*100:.4f}%)", ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Transaction Amount Distributions

We analyze the distribution of transaction amounts on a log-scale and contrast the patterns of legitimate vs laundering funds.

In [ ]:
# Amount density plot comparison
fig, ax = plt.subplots(figsize=(10, 5))
non_laun_amounts = sample_df.loc[sample_df["is_laundering"] == 0, "log_amount"]
laun_amounts = sample_df.loc[sample_df["is_laundering"] == 1, "log_amount"]

ax.hist(non_laun_amounts, bins=50, density=True, alpha=0.5, label="Non-Laundering", color="#5D9CEC")
ax.hist(laun_amounts, bins=50, density=True, alpha=0.5, label="Laundering", color="#FC6E51")
ax.set_title("Log-Transformed Transaction Amount Densities", fontsize=12, fontweight='bold')
ax.set_xlabel("Log(Amount)")
ax.set_ylabel("Density")
ax.legend()
ax.grid(True, linestyle='--', alpha=0.5)
plt.show()

print("=== TRANSACTION AMOUNT STATISTICS BY CLASS ===")
print(sample_df.groupby("is_laundering")["amount"].describe())

## 6. Temporal Distributions

We plot the transaction volume over time and inspect how splits partition the timeline.

In [ ]:
# Convert timestamp to date and count daily volume
sample_df["date"] = sample_df["timestamp"].dt.date
daily_volume = sample_df.groupby("date").size()

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(daily_volume.index, daily_volume.values, marker='o', color='#AC92EC', linewidth=2)
ax.set_title("Daily Transaction volume (Sampled)", fontsize=12, fontweight='bold')
ax.set_xlabel("Date")
ax.set_ylabel("Transaction Count")
ax.grid(True, linestyle='--', alpha=0.5)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("=== TEMPORAL SPLIT RANGES AND FRACTIONS ===")
for split_name, group in sample_df.groupby("split"):
    print(f"Split: {split_name.upper()}")
    print(f"  Rows count:   {len(group):,}")
    print(f"  Min Date:     {group['timestamp'].min()}")
    print(f"  Max Date:     {group['timestamp'].max()}")
    print(f"  Laundering:   {group['is_laundering'].sum()} ({group['is_laundering'].mean()*100:.5f}%)")

## 7. Categorical Feature Analysis

We analyze transaction types (Payment formats), currency pairs, and cross-bank indicators, and compute their associated laundering rates.

In [ ]:
# Payment Format analysis
fmt_counts = sample_df["payment_format"].value_counts()
fmt_laun = sample_df.groupby("payment_format")["is_laundering"].mean()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Left chart: volume
axes[0].bar(fmt_counts.index, fmt_counts.values, color="#48CFAD")
axes[0].set_title("Transaction Count by Payment Format", fontsize=11, fontweight='bold')
axes[0].set_ylabel("Count")
axes[0].tick_params(axis='x', rotation=45)

# Right chart: laundering rate
axes[1].bar(fmt_laun.index, fmt_laun.values * 100, color="#FFCE54")
axes[1].set_title("Laundering Rate by Payment Format (%)", fontsize=11, fontweight='bold')
axes[1].set_ylabel("Laundering Rate (%)")
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Top Currency Pairs
top_currency = sample_df["currency_pair"].value_counts().head(10)
curr_laun_rates = sample_df.groupby("currency_pair")["is_laundering"].mean().loc[top_currency.index]
curr_df = pd.DataFrame({
    "Total Transactions": top_currency,
    "Laundering Rate (%)": curr_laun_rates * 100
})
print("=== TOP 10 CURRENCY PAIRS ===")
print(curr_df)

# Cross bank flag analysis
cross_bank = sample_df.groupby("cross_bank_flag")["is_laundering"].agg(["count", "sum", "mean"])
cross_bank.columns = ["Total Transactions", "Laundering Transactions", "Laundering Rate (%)"]
cross_bank["Laundering Rate (%)"] *= 100
print("\n=== CROSS BANK FLAG ANALYSIS ===")
print(cross_bank)

## 8. Network Degree Distributions (Power Law)

Legitimate and suspicious transactions show typical financial network attributes: most nodes (accounts) are quiet, while a tiny fraction of hubs are highly active.

In [ ]:
sender_deg = sample_df["sender_account_id"].value_counts()
receiver_deg = sample_df["receiver_account_id"].value_counts()

print(f"Unique Senders:   {len(sender_deg):,}")
print(f"Unique Receivers: {len(receiver_deg):,}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(sender_deg.values, bins=np.logspace(0, 4, 50), alpha=0.5, label="Senders", color="#4A89DC", log=True)
ax.hist(receiver_deg.values, bins=np.logspace(0, 4, 50), alpha=0.5, label="Receivers", color="#37BC9B", log=True)
ax.set_xscale("log")
ax.set_title("Account Degree Distribution (Log-Log scale)", fontsize=12, fontweight='bold')
ax.set_xlabel("Number of Transactions (Degree)")
ax.set_ylabel("Number of Accounts (Count)")
ax.grid(True, linestyle='--', alpha=0.4)
ax.legend()
plt.show()

print("=== TOP 10 ACTIVE SENDERS (DEGREE) ===")
print(sender_deg.head(10))

print("\n=== TOP 10 ACTIVE RECEIVERS (DEGREE) ===")
print(receiver_deg.head(10))